In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
# Importação do SMOTE da biblioteca imblearn
from imblearn.over_sampling import SMOTE

# ==========================================
# 1. CARREGAMENTO E LIMPEZA DOS DADOS
# ==========================================

df = pd.read_csv('credit_data.csv')

# Limpeza dos caracteres especiais nos nomes das colunas
df.columns = df.columns.str.replace('#', '').str.strip()

# Tratamento de idades inconsistentes ou nulas
df.loc[df['age'] < 0, 'age'] = np.nan
df['age'] = df['age'].fillna(df['age'].mean())

# Separando variáveis preditoras (X) e alvo (y)
X = df[['income', 'age', 'loan']].values
y = df['cdefault'].values

print("--- Proporção ANTES do SMOTE (Total) ---")
print(f"Bons pagadores (Classe 0): {np.bincount(y)[0]}")
print(f"Inadimplentes  (Classe 1): {np.bincount(y)[1]}")

# ==========================================
# 2. DIVISÃO EM TREINO E TESTE
# ==========================================
# Regra de ouro: Dividimos antes para aplicar o SMOTE APENAS no treino.
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ==========================================
# 3. PADRONIZAÇÃO DOS DADOS
# ==========================================
# O SMOTE usa o algoritmo KNN (K-Vizinhos Próximos) internamente para criar os dados sintéticos.
# Por isso, a padronização dos dados ANTES do SMOTE é obrigatória.
scaler = StandardScaler()
X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)

# ==========================================
# 4. APLICAÇÃO DO SMOTE (SOBREAMOSTRAGEM)
# ==========================================

# Criamos o objeto SMOTE (por padrão, ele iguala a quantidade das duas classes)
smote = SMOTE(random_state=42)

# Geramos os novos dados sintéticos de treino
X_treino_balanceado, y_treino_balanceado = smote.fit_resample(X_treino_scaled, y_treino)

print("\n--- Proporção APÓS o SMOTE (Apenas no Treino) ---")
print(f"Bons pagadores (Classe 0): {np.bincount(y_treino_balanceado)[0]}")
print(f"Inadimplentes  (Classe 1) [Sintéticos + Reais]: {np.bincount(y_treino_balanceado)[1]}")

# ==========================================
# 5. TREINAMENTO DO MODELO (NAIVE BAYES)
# ==========================================

modelo_nb = GaussianNB()
# Treinando o modelo com a base ampliada pelo SMOTE
modelo_nb.fit(X_treino_balanceado, y_treino_balanceado)

# Previsões no conjunto de teste original (mundo real)
y_pred = modelo_nb.predict(X_teste_scaled)

# ==========================================
# 6. AVALIAÇÃO DO MODELO
# ==========================================

print("\n----- AVALIAÇÃO DO MODELO COM SMOTE -----")
print(f"Acurácia: {accuracy_score(y_teste, y_pred) * 100:.2f}%")
print("\nMatriz de Confusão:")
print(confusion_matrix(y_teste, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_teste, y_pred))

# ==========================================
# 7. INFERÊNCIA / PROBABILIDADE PARA UM NOVO DADO
# ==========================================

print("\n----- PREVISÃO PARA NOVO CLIENTE -----")

# Novo cliente de teste: Renda: 45.000 | Idade: 22 | Empréstimo solicitado: 7.000
novo_cliente = np.array([[45000, 22, 7000]])

# Padronizando o novo dado com o mesmo scaler do treino
novo_cliente_scaled = scaler.transform(novo_cliente)

# Predição direta e cálculo das probabilidades
previsao = modelo_nb.predict(novo_cliente_scaled)
probabilidades = modelo_nb.predict_proba(novo_cliente_scaled)

print(f"Dados informados -> Renda: {novo_cliente[0][0]}, Idade: {novo_cliente[0][1]}, Empréstimo: {novo_cliente[0][2]}")
print(f"Probabilidade de Pagar (Classe 0): {probabilidades[0][0] * 100:.2f}%")
print(f"Probabilidade de Não Pagar (Classe 1): {probabilidades[0][1] * 100:.2f}%")

if previsao[0] == 1:
    print("Resultado Estratégico: Crédito Recusado (Risco de Inadimplência baseado em perfis gerados).")
else:
    print("Resultado Estratégico: Crédito Aprovado.")

--- Proporção ANTES do SMOTE (Total) ---
Bons pagadores (Classe 0): 1717
Inadimplentes  (Classe 1): 283

--- Proporção APÓS o SMOTE (Apenas no Treino) ---
Bons pagadores (Classe 0): 1374
Inadimplentes  (Classe 1) [Sintéticos + Reais]: 1374

----- AVALIAÇÃO DO MODELO COM SMOTE -----
Acurácia: 87.25%

Matriz de Confusão:
[[299  44]
 [  7  50]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.98      0.87      0.92       343
           1       0.53      0.88      0.66        57

    accuracy                           0.87       400
   macro avg       0.75      0.87      0.79       400
weighted avg       0.91      0.87      0.88       400


----- PREVISÃO PARA NOVO CLIENTE -----
Dados informados -> Renda: 45000, Idade: 22, Empréstimo: 7000
Probabilidade de Pagar (Classe 0): 6.35%
Probabilidade de Não Pagar (Classe 1): 93.65%
Resultado Estratégico: Crédito Recusado (Risco de Inadimplência baseado em perfis gerados).
